# Experiment: kaggle_submission_exp_003_perch

Objective:
- Build a Kaggle-ready submission notebook for the Perch-based `exp_003` path.
- Reuse the cached `perch_meta` train-side outputs to fit priors and classwise probes.
- Run offline Perch inference on hidden `test_soundscapes` and save `/kaggle/working/submission.csv`.


## Attach These Inputs On Kaggle

- Competition data: `BirdCLEF+ 2026`
- Offline TensorFlow wheels dataset from the `bc26-tensorflow-2-20-0` packaging path
- Dataset or model containing `perch_v2_cpu`
- Dataset containing `full_perch_meta.parquet` and `full_perch_arrays.npz`

If multiple attached datasets contain similar files, set the `*_DATASET_HINT` values in the first code cell below.


In [ ]:
# Cell 0 — Resolve offline wheels and install TensorFlow without internet
from __future__ import annotations

import importlib.metadata
import json
import subprocess
import sys
from pathlib import Path

INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working/perch_exp003')
WORK_ROOT.mkdir(parents=True, exist_ok=True)

WHEELS_DATASET_HINT = None        # Example: 'bc26-tensorflow-2-20-0'
PERCH_MODEL_DATASET_HINT = None   # Example: 'bird-vocalization-classifier-tensorflow2-perch-v2-cpu-v1'
PERCH_META_DATASET_HINT = None    # Example: 'perch-meta'
VERBOSE = True
TF_TARGET_VERSION = '2.20.0'
FORCE_TF_REINSTALL = True


def _matches_hint(path: Path, hint: str | None) -> bool:
    if hint is None:
        return True
    hint = hint.lower()
    return hint in str(path).lower()


def find_files(pattern: str, hint: str | None = None):
    return sorted([path for path in INPUT_ROOT.rglob(pattern) if _matches_hint(path, hint)])


def resolve_single_file(pattern: str, hint: str | None = None) -> Path:
    matches = find_files(pattern, hint=hint)
    if not matches:
        raise FileNotFoundError(f'Could not find {pattern} under {INPUT_ROOT} (hint={hint})')
    return matches[0]


def get_installed_tf_version() -> str | None:
    try:
        return importlib.metadata.version('tensorflow')
    except importlib.metadata.PackageNotFoundError:
        return None


def install_offline_tensorflow():
    installed_version = get_installed_tf_version()
    needs_install = FORCE_TF_REINSTALL or installed_version != TF_TARGET_VERSION

    if not needs_install:
        print(f'TensorFlow {installed_version} already matches the target version.')
        return

    tensorboard_wheel = resolve_single_file(f'tensorboard-{TF_TARGET_VERSION}-*.whl', hint=WHEELS_DATASET_HINT)
    tensorflow_wheel = resolve_single_file(f'tensorflow-{TF_TARGET_VERSION}-*.whl', hint=WHEELS_DATASET_HINT)

    print('Installing offline wheels:')
    print('  installed tensorflow:', installed_version)
    print('  target tensorflow   :', TF_TARGET_VERSION)
    print('  ', tensorboard_wheel)
    print('  ', tensorflow_wheel)

    for wheel in [tensorboard_wheel, tensorflow_wheel]:
        subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', str(wheel)],
            check=True,
        )

install_offline_tensorflow()
print(json.dumps({
    'input_root': str(INPUT_ROOT),
    'work_root': str(WORK_ROOT),
    'wheels_hint': WHEELS_DATASET_HINT,
    'model_hint': PERCH_MODEL_DATASET_HINT,
    'perch_meta_hint': PERCH_META_DATASET_HINT,
    'tf_target_version': TF_TARGET_VERSION,
    'force_tf_reinstall': FORCE_TF_REINSTALL,
}, indent=2))


In [ ]:
# Cell 1 — Imports and runtime config
import gc
import io
import json
import os
import re
import tarfile
import warnings
from pathlib import Path

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['CUDA_VISIBLE_DEVICES'] = ''

import numpy as np
import pandas as pd
import soundfile as sf
import tensorflow as tf
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler

try:
    from tqdm.auto import tqdm
except ModuleNotFoundError:
    def tqdm(iterable=None, *args, **kwargs):
        return iterable if iterable is not None else []

warnings.filterwarnings('ignore')
tf.experimental.numpy.experimental_enable_numpy_behavior()

MODE = 'submit'
SR = 32000
WINDOW_SEC = 5
WINDOW_SAMPLES = SR * WINDOW_SEC
FILE_SAMPLES = 60 * SR
N_WINDOWS = 12

CFG = {
    'mode': MODE,
    'verbose': False,
    'batch_files': 16,
    'best_fusion': {
        'lambda_event': 0.4,
        'lambda_texture': 1.0,
        'lambda_proxy_texture': 0.8,
        'smooth_texture': 0.35,
    },
    'frozen_best_probe': {
        'pca_dim': 64,
        'min_pos': 8,
        'C': 0.50,
        'alpha': 0.40,
    },
}

print('TensorFlow:', tf.__version__)


In [ ]:
# Cell 2 — Resolve competition data, model, and cached train-side Perch outputs

def resolve_competition_dir() -> Path:
    candidates = [
        INPUT_ROOT / 'competitions' / 'birdclef-2026',
        INPUT_ROOT / 'birdclef-2026',
    ]
    for candidate in candidates:
        if (candidate / 'sample_submission.csv').exists():
            return candidate
    for candidate in INPUT_ROOT.rglob('sample_submission.csv'):
        if candidate.parent.name == 'birdclef-2026' or 'birdclef-2026' in str(candidate.parent):
            return candidate.parent
    raise FileNotFoundError('Could not resolve competition directory with sample_submission.csv')


def resolve_perch_meta_paths() -> tuple[Path, Path]:
    meta_candidates = find_files('full_perch_meta.parquet', hint=PERCH_META_DATASET_HINT)
    npz_candidates = find_files('full_perch_arrays.npz', hint=PERCH_META_DATASET_HINT)
    if not meta_candidates or not npz_candidates:
        raise FileNotFoundError('Could not find cached perch_meta files under /kaggle/input')
    return meta_candidates[0], npz_candidates[0]


def resolve_model_dir() -> Path:
    dir_candidates = []
    for saved_model in INPUT_ROOT.rglob('saved_model.pb'):
        model_dir = saved_model.parent
        if (model_dir / 'assets' / 'labels.csv').exists() and _matches_hint(model_dir, PERCH_MODEL_DATASET_HINT):
            dir_candidates.append(model_dir)
    if dir_candidates:
        return sorted(dir_candidates)[0]

    tar_candidates = [
        path for path in INPUT_ROOT.rglob('*.tar.gz')
        if 'perch' in path.name.lower() and _matches_hint(path, PERCH_MODEL_DATASET_HINT)
    ]
    if not tar_candidates:
        raise FileNotFoundError('Could not find Perch model directory or archive under /kaggle/input')

    archive_path = sorted(tar_candidates)[0]
    extract_dir = WORK_ROOT / 'resolved_perch_model'
    if not (extract_dir / 'saved_model.pb').exists():
        extract_dir.mkdir(parents=True, exist_ok=True)
        with tarfile.open(archive_path, 'r:gz') as tf_archive:
            tf_archive.extractall(extract_dir)
    return extract_dir

BASE = resolve_competition_dir()
PERCH_META_PATH, PERCH_ARRAYS_PATH = resolve_perch_meta_paths()
MODEL_DIR = resolve_model_dir()

birdclassifier = tf.saved_model.load(str(MODEL_DIR))
infer_fn = birdclassifier.signatures['serving_default']

print('Competition dir:', BASE)
print('Perch meta parquet:', PERCH_META_PATH)
print('Perch arrays npz:', PERCH_ARRAYS_PATH)
print('Model dir:', MODEL_DIR)


In [ ]:
# Cell 3 — Load taxonomy, trusted soundscape labels, and scientific-name mapping

taxonomy = pd.read_csv(BASE / 'taxonomy.csv')
sample_sub = pd.read_csv(BASE / 'sample_submission.csv')
soundscape_labels = pd.read_csv(BASE / 'train_soundscapes_labels.csv')

PRIMARY_LABELS = sample_sub.columns[1:].tolist()
N_CLASSES = len(PRIMARY_LABELS)

label_to_idx = {label: idx for idx, label in enumerate(PRIMARY_LABELS)}
TEXTURE_TAXA = {'Amphibia', 'Insecta'}
FNAME_RE = re.compile(r'BC2026_(?:Train|Test)_(\d+)_(S\d+)_(\d{8})_(\d{6})\.ogg')


def parse_soundscape_labels(x):
    if pd.isna(x):
        return []
    return [token.strip() for token in str(x).split(';') if token.strip()]


def union_labels(series):
    return sorted(set(label for item in series for label in parse_soundscape_labels(item)))


def parse_soundscape_filename(name: str):
    match = FNAME_RE.match(name)
    if not match:
        return {
            'file_id': None,
            'site': None,
            'date': pd.NaT,
            'time_utc': None,
            'hour_utc': -1,
            'month': -1,
        }
    file_id, site, ymd, hms = match.groups()
    dt = pd.to_datetime(ymd, format='%Y%m%d', errors='coerce')
    return {
        'file_id': file_id,
        'site': site,
        'date': dt,
        'time_utc': hms,
        'hour_utc': int(hms[:2]),
        'month': int(dt.month) if pd.notna(dt) else -1,
    }


def macro_auc_skip_empty(y_true, y_score):
    keep = y_true.sum(axis=0) > 0
    return float(roc_auc_score(y_true[:, keep], y_score[:, keep], average='macro'))

sc_clean = (
    soundscape_labels
    .groupby(['filename', 'start', 'end'])['primary_label']
    .apply(union_labels)
    .reset_index(name='label_list')
)
sc_clean['start_sec'] = pd.to_timedelta(sc_clean['start']).dt.total_seconds().astype(int)
sc_clean['end_sec'] = pd.to_timedelta(sc_clean['end']).dt.total_seconds().astype(int)
sc_clean['row_id'] = sc_clean['filename'].str.replace('.ogg', '', regex=False) + '_' + sc_clean['end_sec'].astype(str)
sc_meta = sc_clean['filename'].apply(parse_soundscape_filename).apply(pd.Series)
sc_clean = pd.concat([sc_clean, sc_meta], axis=1)

windows_per_file = sc_clean.groupby('filename').size()
full_files = sorted(windows_per_file[windows_per_file == N_WINDOWS].index.tolist())
sc_clean['file_fully_labeled'] = sc_clean['filename'].isin(full_files)

Y_SC = np.zeros((len(sc_clean), N_CLASSES), dtype=np.uint8)
for row_idx, labels in enumerate(sc_clean['label_list']):
    indices = [label_to_idx[label] for label in labels if label in label_to_idx]
    if indices:
        Y_SC[row_idx, indices] = 1

full_truth = (
    sc_clean[sc_clean['file_fully_labeled']]
    .sort_values(['filename', 'end_sec'])
    .reset_index(drop=False)
)


In [ ]:
# Cell 4 — Continue mapping setup using Perch labels from the resolved model directory
bc_labels = (
    pd.read_csv(MODEL_DIR / 'assets' / 'labels.csv')
    .reset_index()
    .rename(columns={'index': 'bc_index', 'inat2024_fsd50k': 'scientific_name'})
)

NO_LABEL_INDEX = len(bc_labels)
MANUAL_SCIENTIFIC_NAME_MAP = {}

taxonomy = taxonomy.copy()
taxonomy['primary_label'] = taxonomy['primary_label'].astype(str)
taxonomy['scientific_name_lookup'] = taxonomy['scientific_name'].replace(MANUAL_SCIENTIFIC_NAME_MAP)

bc_lookup = bc_labels.rename(columns={'scientific_name': 'scientific_name_lookup'})
mapping = taxonomy.merge(
    bc_lookup[['scientific_name_lookup', 'bc_index']],
    on='scientific_name_lookup',
    how='left',
)
mapping['bc_index'] = mapping['bc_index'].fillna(NO_LABEL_INDEX).astype(int)

label_to_bc_index = mapping.set_index('primary_label')['bc_index']
BC_INDICES = np.array([int(label_to_bc_index.loc[label]) for label in PRIMARY_LABELS], dtype=np.int32)
MAPPED_MASK = BC_INDICES != NO_LABEL_INDEX
MAPPED_POS = np.where(MAPPED_MASK)[0].astype(np.int32)
UNMAPPED_POS = np.where(~MAPPED_MASK)[0].astype(np.int32)
MAPPED_BC_INDICES = BC_INDICES[MAPPED_MASK].astype(np.int32)

CLASS_NAME_MAP = taxonomy.set_index('primary_label')['class_name'].to_dict()
ACTIVE_CLASSES = [PRIMARY_LABELS[i] for i in np.where(Y_SC.sum(axis=0) > 0)[0]]

idx_active_texture = np.array(
    [label_to_idx[label] for label in ACTIVE_CLASSES if CLASS_NAME_MAP.get(label) in TEXTURE_TAXA],
    dtype=np.int32,
)
idx_active_event = np.array(
    [label_to_idx[label] for label in ACTIVE_CLASSES if CLASS_NAME_MAP.get(label) not in TEXTURE_TAXA],
    dtype=np.int32,
)
idx_mapped_active_texture = idx_active_texture[MAPPED_MASK[idx_active_texture]]
idx_mapped_active_event = idx_active_event[MAPPED_MASK[idx_active_event]]
idx_unmapped_active_texture = idx_active_texture[~MAPPED_MASK[idx_active_texture]]
idx_unmapped_active_event = idx_active_event[~MAPPED_MASK[idx_active_event]]
idx_unmapped_inactive = np.array(
    [i for i in UNMAPPED_POS if PRIMARY_LABELS[i] not in ACTIVE_CLASSES],
    dtype=np.int32,
)

unmapped_df = mapping[mapping['bc_index'] == NO_LABEL_INDEX].copy()
unmapped_non_sonotype = unmapped_df[
    ~unmapped_df['primary_label'].astype(str).str.contains('son', na=False)
].copy()


def get_genus_hits(scientific_name):
    genus = str(scientific_name).split()[0]
    hits = bc_labels[
        bc_labels['scientific_name'].astype(str).str.match(rf'^{re.escape(genus)}\s', na=False)
    ].copy()
    return genus, hits

proxy_map = {}
for _, row in unmapped_non_sonotype.iterrows():
    target = row['primary_label']
    genus, hits = get_genus_hits(row['scientific_name'])
    if len(hits) > 0:
        proxy_map[target] = {
            'target_scientific_name': row['scientific_name'],
            'genus': genus,
            'bc_indices': hits['bc_index'].astype(int).tolist(),
            'proxy_scientific_names': hits['scientific_name'].tolist(),
        }

SELECTED_PROXY_TARGETS = sorted([
    target for target in proxy_map.keys()
    if CLASS_NAME_MAP.get(target) == 'Amphibia'
])
selected_proxy_pos = np.array([label_to_idx[label] for label in SELECTED_PROXY_TARGETS], dtype=np.int32)
selected_proxy_pos_to_bc = {
    label_to_idx[target]: np.array(proxy_map[target]['bc_indices'], dtype=np.int32)
    for target in SELECTED_PROXY_TARGETS
}
idx_selected_proxy_active_texture = np.intersect1d(selected_proxy_pos, idx_active_texture)
idx_selected_prioronly_active_texture = np.setdiff1d(idx_unmapped_active_texture, selected_proxy_pos)
idx_selected_prioronly_active_event = np.setdiff1d(idx_unmapped_active_event, selected_proxy_pos)

print(json.dumps({
    'full_files': int(len(full_files)),
    'active_classes_in_full_windows': int((Y_SC[full_truth['index'].to_numpy()].sum(axis=0) > 0).sum()),
    'mapped_classes': int(MAPPED_MASK.sum()),
    'unmapped_classes': int((~MAPPED_MASK).sum()),
    'active_texture_classes': int(len(idx_active_texture)),
}, indent=2))


In [ ]:
# Cell 5 — Metadata priors, probe features, and Perch inference helpers
BEST = CFG['best_fusion']
BEST_PROBE = CFG['frozen_best_probe']


def fit_prior_tables(prior_df, y_prior):
    prior_df = prior_df.reset_index(drop=True)
    global_p = y_prior.mean(axis=0).astype(np.float32)

    site_keys = sorted(prior_df['site'].dropna().astype(str).unique().tolist())
    site_to_i = {key: idx for idx, key in enumerate(site_keys)}
    site_n = np.zeros(len(site_keys), dtype=np.float32)
    site_p = np.zeros((len(site_keys), y_prior.shape[1]), dtype=np.float32)
    for site in site_keys:
        idx = site_to_i[site]
        mask = prior_df['site'].astype(str).values == site
        site_n[idx] = mask.sum()
        site_p[idx] = y_prior[mask].mean(axis=0)

    hour_keys = sorted(prior_df['hour_utc'].dropna().astype(int).unique().tolist())
    hour_to_i = {hour: idx for idx, hour in enumerate(hour_keys)}
    hour_n = np.zeros(len(hour_keys), dtype=np.float32)
    hour_p = np.zeros((len(hour_keys), y_prior.shape[1]), dtype=np.float32)
    for hour in hour_keys:
        idx = hour_to_i[hour]
        mask = prior_df['hour_utc'].astype(int).values == hour
        hour_n[idx] = mask.sum()
        hour_p[idx] = y_prior[mask].mean(axis=0)

    sh_to_i = {}
    sh_n_list = []
    sh_p_list = []
    for (site, hour), group_idx in prior_df.groupby(['site', 'hour_utc']).groups.items():
        sh_to_i[(str(site), int(hour))] = len(sh_n_list)
        group_idx = np.array(list(group_idx))
        sh_n_list.append(len(group_idx))
        sh_p_list.append(y_prior[group_idx].mean(axis=0))

    sh_n = np.array(sh_n_list, dtype=np.float32)
    sh_p = np.stack(sh_p_list).astype(np.float32) if len(sh_p_list) else np.zeros((0, y_prior.shape[1]), dtype=np.float32)

    return {
        'global_p': global_p,
        'site_to_i': site_to_i,
        'site_n': site_n,
        'site_p': site_p,
        'hour_to_i': hour_to_i,
        'hour_n': hour_n,
        'hour_p': hour_p,
        'sh_to_i': sh_to_i,
        'sh_n': sh_n,
        'sh_p': sh_p,
    }


def prior_logits_from_tables(sites, hours, tables, eps=1e-4):
    n_rows = len(sites)
    p = np.repeat(tables['global_p'][None, :], n_rows, axis=0).astype(np.float32, copy=True)

    site_idx = np.fromiter((tables['site_to_i'].get(str(site), -1) for site in sites), dtype=np.int32, count=n_rows)
    hour_idx = np.fromiter((tables['hour_to_i'].get(int(hour), -1) if int(hour) >= 0 else -1 for hour in hours), dtype=np.int32, count=n_rows)
    sh_idx = np.fromiter(
        (tables['sh_to_i'].get((str(site), int(hour)), -1) if int(hour) >= 0 else -1 for site, hour in zip(sites, hours)),
        dtype=np.int32,
        count=n_rows,
    )

    valid = hour_idx >= 0
    if valid.any():
        nh = tables['hour_n'][hour_idx[valid]][:, None]
        wh = nh / (nh + 8.0)
        p[valid] = wh * tables['hour_p'][hour_idx[valid]] + (1.0 - wh) * p[valid]

    valid = site_idx >= 0
    if valid.any():
        ns = tables['site_n'][site_idx[valid]][:, None]
        ws = ns / (ns + 8.0)
        p[valid] = ws * tables['site_p'][site_idx[valid]] + (1.0 - ws) * p[valid]

    valid = sh_idx >= 0
    if valid.any():
        nsh = tables['sh_n'][sh_idx[valid]][:, None]
        wsh = nsh / (nsh + 4.0)
        p[valid] = wsh * tables['sh_p'][sh_idx[valid]] + (1.0 - wsh) * p[valid]

    np.clip(p, eps, 1.0 - eps, out=p)
    return (np.log(p) - np.log1p(-p)).astype(np.float32, copy=False)


def smooth_cols_fixed12(scores, cols, alpha=0.35):
    if alpha <= 0 or len(cols) == 0:
        return scores.copy()
    s = scores.copy()
    assert len(s) % N_WINDOWS == 0
    view = s.reshape(-1, N_WINDOWS, s.shape[1])
    x = view[:, :, cols]
    prev_x = np.concatenate([x[:, :1, :], x[:, :-1, :]], axis=1)
    next_x = np.concatenate([x[:, 1:, :], x[:, -1:, :]], axis=1)
    view[:, :, cols] = (1.0 - alpha) * x + 0.5 * alpha * (prev_x + next_x)
    return s


def seq_features_1d(v):
    assert len(v) % N_WINDOWS == 0
    x = v.reshape(-1, N_WINDOWS)
    prev_v = np.concatenate([x[:, :1], x[:, :-1]], axis=1).reshape(-1)
    next_v = np.concatenate([x[:, 1:], x[:, -1:]], axis=1).reshape(-1)
    mean_v = np.repeat(x.mean(axis=1), N_WINDOWS)
    max_v = np.repeat(x.max(axis=1), N_WINDOWS)
    return prev_v, next_v, mean_v, max_v


def fuse_scores_with_tables(base_scores, sites, hours, tables):
    scores = base_scores.copy()
    prior = prior_logits_from_tables(sites, hours, tables)

    if len(idx_mapped_active_event):
        scores[:, idx_mapped_active_event] += BEST['lambda_event'] * prior[:, idx_mapped_active_event]
    if len(idx_mapped_active_texture):
        scores[:, idx_mapped_active_texture] += BEST['lambda_texture'] * prior[:, idx_mapped_active_texture]
    if len(idx_selected_proxy_active_texture):
        scores[:, idx_selected_proxy_active_texture] += BEST['lambda_proxy_texture'] * prior[:, idx_selected_proxy_active_texture]
    if len(idx_selected_prioronly_active_event):
        scores[:, idx_selected_prioronly_active_event] = BEST['lambda_event'] * prior[:, idx_selected_prioronly_active_event]
    if len(idx_selected_prioronly_active_texture):
        scores[:, idx_selected_prioronly_active_texture] = BEST['lambda_texture'] * prior[:, idx_selected_prioronly_active_texture]
    if len(idx_unmapped_inactive):
        scores[:, idx_unmapped_inactive] = -8.0

    scores = smooth_cols_fixed12(scores, idx_active_texture, alpha=BEST['smooth_texture'])
    return scores.astype(np.float32, copy=False), prior


def build_oof_base_prior(scores_full_raw, meta_full, sc_clean_df, y_sc, n_splits=5):
    groups_full = meta_full['filename'].to_numpy()
    gkf = GroupKFold(n_splits=n_splits)

    oof_base = np.zeros_like(scores_full_raw, dtype=np.float32)
    oof_prior = np.zeros_like(scores_full_raw, dtype=np.float32)

    for _, (train_idx, valid_idx) in enumerate(gkf.split(scores_full_raw, groups=groups_full), 1):
        valid_files = set(meta_full.iloc[valid_idx]['filename'].tolist())
        prior_mask = ~sc_clean_df['filename'].isin(valid_files).values
        prior_df_fold = sc_clean_df.loc[prior_mask].reset_index(drop=True)
        y_prior_fold = y_sc[prior_mask]
        tables = fit_prior_tables(prior_df_fold, y_prior_fold)

        valid_base, valid_prior = fuse_scores_with_tables(
            scores_full_raw[valid_idx],
            sites=meta_full.iloc[valid_idx]['site'].to_numpy(),
            hours=meta_full.iloc[valid_idx]['hour_utc'].to_numpy(),
            tables=tables,
        )
        oof_base[valid_idx] = valid_base
        oof_prior[valid_idx] = valid_prior

    return oof_base, oof_prior


def build_class_features(emb_proj, raw_col, prior_col, base_col):
    prev_base, next_base, mean_base, max_base = seq_features_1d(base_col)
    feats = np.concatenate([
        emb_proj,
        raw_col[:, None],
        prior_col[:, None],
        base_col[:, None],
        prev_base[:, None],
        next_base[:, None],
        mean_base[:, None],
        max_base[:, None],
    ], axis=1)
    return feats.astype(np.float32, copy=False)


def read_soundscape_60s(path):
    y, sr = sf.read(path, dtype='float32', always_2d=False)
    if y.ndim == 2:
        y = y.mean(axis=1)
    if sr != SR:
        raise ValueError(f'Unexpected sample rate {sr} in {path}; expected {SR}')
    if len(y) < FILE_SAMPLES:
        y = np.pad(y, (0, FILE_SAMPLES - len(y)))
    elif len(y) > FILE_SAMPLES:
        y = y[:FILE_SAMPLES]
    return y


def infer_perch_with_embeddings(paths, batch_files=16, verbose=True, proxy_reduce='max'):
    paths = [Path(p) for p in paths]
    n_files = len(paths)
    n_rows = n_files * N_WINDOWS

    row_ids = np.empty(n_rows, dtype=object)
    filenames = np.empty(n_rows, dtype=object)
    sites = np.empty(n_rows, dtype=object)
    hours = np.empty(n_rows, dtype=np.int16)
    scores = np.zeros((n_rows, N_CLASSES), dtype=np.float32)
    embeddings = np.zeros((n_rows, 1536), dtype=np.float32)

    write_row = 0
    iterator = range(0, n_files, batch_files)
    if verbose:
        iterator = tqdm(iterator, total=(n_files + batch_files - 1) // batch_files, desc='Perch batches')

    for start in iterator:
        batch_paths = paths[start:start + batch_files]
        batch_n = len(batch_paths)
        x = np.empty((batch_n * N_WINDOWS, WINDOW_SAMPLES), dtype=np.float32)
        batch_row_start = write_row
        x_pos = 0

        for path in batch_paths:
            y = read_soundscape_60s(path)
            x[x_pos:x_pos + N_WINDOWS] = y.reshape(N_WINDOWS, WINDOW_SAMPLES)

            meta = parse_soundscape_filename(path.name)
            stem = path.stem
            row_ids[write_row:write_row + N_WINDOWS] = [f'{stem}_{t}' for t in range(5, 65, 5)]
            filenames[write_row:write_row + N_WINDOWS] = path.name
            sites[write_row:write_row + N_WINDOWS] = meta['site']
            hours[write_row:write_row + N_WINDOWS] = int(meta['hour_utc'])

            x_pos += N_WINDOWS
            write_row += N_WINDOWS

        outputs = infer_fn(inputs=tf.convert_to_tensor(x))
        logits = outputs['label'].numpy().astype(np.float32, copy=False)
        emb = outputs['embedding'].numpy().astype(np.float32, copy=False)

        scores[batch_row_start:write_row, MAPPED_POS] = logits[:, MAPPED_BC_INDICES]
        embeddings[batch_row_start:write_row] = emb

        for pos, bc_idx_arr in selected_proxy_pos_to_bc.items():
            sub = logits[:, bc_idx_arr]
            proxy_score = sub.max(axis=1) if proxy_reduce == 'max' else sub.mean(axis=1)
            scores[batch_row_start:write_row, pos] = proxy_score.astype(np.float32)

        del x, outputs, logits, emb
        gc.collect()

    meta_df = pd.DataFrame({
        'row_id': row_ids,
        'filename': filenames,
        'site': sites,
        'hour_utc': hours,
    })
    return meta_df, scores, embeddings


In [ ]:
# Cell 6 — Load cached train-side Perch outputs and fit the final soundscape stack
meta_full = pd.read_parquet(PERCH_META_PATH)
perch_arrays = np.load(PERCH_ARRAYS_PATH)
scores_full_raw = perch_arrays['scores_full_raw'].astype(np.float32)
emb_full = perch_arrays['emb_full'].astype(np.float32)

full_truth_aligned = full_truth.set_index('row_id').loc[meta_full['row_id']].reset_index()
Y_FULL = Y_SC[full_truth_aligned['index'].to_numpy()]

assert np.all(full_truth_aligned['filename'].values == meta_full['filename'].values)
assert np.all(full_truth_aligned['row_id'].values == meta_full['row_id'].values)

oof_base, oof_prior = build_oof_base_prior(
    scores_full_raw=scores_full_raw,
    meta_full=meta_full,
    sc_clean_df=sc_clean,
    y_sc=Y_SC,
    n_splits=5,
)

final_prior_tables = fit_prior_tables(sc_clean.reset_index(drop=True), Y_SC)
emb_scaler = StandardScaler()
emb_full_scaled = emb_scaler.fit_transform(emb_full)

n_comp = min(int(BEST_PROBE['pca_dim']), emb_full_scaled.shape[0] - 1, emb_full_scaled.shape[1])
emb_pca = PCA(n_components=n_comp)
Z_FULL = emb_pca.fit_transform(emb_full_scaled).astype(np.float32)

full_pos_counts = Y_FULL.sum(axis=0)
probe_class_idx = np.where(full_pos_counts >= int(BEST_PROBE['min_pos']))[0].astype(np.int32)
probe_models = {}

for cls_idx in tqdm(probe_class_idx, desc='Training final classwise probes', disable=not CFG['verbose']):
    y = Y_FULL[:, cls_idx]
    if y.sum() == 0 or y.sum() == len(y):
        continue

    X_cls = build_class_features(
        Z_FULL,
        raw_col=scores_full_raw[:, cls_idx],
        prior_col=oof_prior[:, cls_idx],
        base_col=oof_base[:, cls_idx],
    )

    clf = LogisticRegression(
        C=float(BEST_PROBE['C']),
        max_iter=400,
        solver='liblinear',
        class_weight='balanced',
    )
    clf.fit(X_cls, y)
    probe_models[cls_idx] = clf

print(json.dumps({
    'trusted_windows': int(len(meta_full)),
    'raw_local_auc': macro_auc_skip_empty(Y_FULL, scores_full_raw),
    'baseline_oof_auc': macro_auc_skip_empty(Y_FULL, oof_base),
    'modeled_probe_classes': int(len(probe_models)),
    'pca_dim': int(n_comp),
}, indent=2))


In [ ]:
# Cell 7 — Run Perch on hidden test soundscapes

def resolve_test_soundscape_paths(base: Path, sample_sub: pd.DataFrame) -> list[Path]:
    expected_stems = {str(row_id).rsplit('_', 1)[0] for row_id in sample_sub['row_id'].tolist()}
    audio_suffixes = {'.ogg', '.wav', '.flac', '.mp3'}

    search_roots = [
        base / 'test_soundscapes',
        base / 'test_audio',
        base,
    ]

    candidates: dict[str, Path] = {}
    for root in search_roots:
        if not root.exists():
            continue
        for path in root.rglob('*'):
            if path.is_file() and path.suffix.lower() in audio_suffixes and path.stem in expected_stems:
                candidates[path.stem] = path

    if not candidates:
        for path in INPUT_ROOT.rglob('*'):
            if path.is_file() and path.suffix.lower() in audio_suffixes and path.stem in expected_stems:
                candidates[path.stem] = path

    return [candidates[stem] for stem in sorted(candidates)]


used_test_fallback = False
test_paths = resolve_test_soundscape_paths(BASE, sample_sub)
expected_test_stems = {str(row_id).rsplit('_', 1)[0] for row_id in sample_sub['row_id'].tolist()}
found_test_stems = {path.stem for path in test_paths}
missing_test_stems = sorted(expected_test_stems - found_test_stems)

if not test_paths:
    train_fallback = BASE / 'train_soundscapes'
    fallback_paths = []
    if train_fallback.exists():
        fallback_paths = sorted(train_fallback.rglob('*.ogg'))[:3]
    if fallback_paths:
        used_test_fallback = True
        test_paths = fallback_paths
        print('Warning: hidden test files not found; using a small train fallback for a dry run.')
    else:
        raise FileNotFoundError(
            f'No soundscape files found for inference. BASE={BASE}. '
            'The notebook could not match any audio file to sample_submission row_ids.'
        )

meta_test, scores_test_raw, emb_test = infer_perch_with_embeddings(
    test_paths,
    batch_files=int(CFG['batch_files']),
    verbose=bool(CFG['verbose']),
    proxy_reduce='max',
)

print('Test files:', len(test_paths))
print('First test path:', test_paths[0])
print('used_test_fallback:', used_test_fallback)
print('matched test stems:', len(found_test_stems), '/', len(expected_test_stems))
print('missing test stems sample:', missing_test_stems[:5])
print('meta_test:', meta_test.shape)
print('scores_test_raw:', scores_test_raw.shape)
print('emb_test:', emb_test.shape)


In [ ]:
# Cell 8 — Build submission from priors + probes

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

test_base_scores, test_prior_scores = fuse_scores_with_tables(
    scores_test_raw,
    sites=meta_test['site'].to_numpy(),
    hours=meta_test['hour_utc'].to_numpy(),
    tables=final_prior_tables,
)

emb_test_scaled = emb_scaler.transform(emb_test)
Z_TEST = emb_pca.transform(emb_test_scaled).astype(np.float32)

final_test_scores = test_base_scores.copy()
for cls_idx, clf in tqdm(probe_models.items(), desc='Applying classwise probes to test', disable=not CFG['verbose']):
    X_cls_test = build_class_features(
        Z_TEST,
        raw_col=scores_test_raw[:, cls_idx],
        prior_col=test_prior_scores[:, cls_idx],
        base_col=test_base_scores[:, cls_idx],
    )
    pred = clf.decision_function(X_cls_test).astype(np.float32)
    final_test_scores[:, cls_idx] = (
        (1.0 - float(BEST_PROBE['alpha'])) * test_base_scores[:, cls_idx] +
        float(BEST_PROBE['alpha']) * pred
    )

score_df = pd.DataFrame(sigmoid(final_test_scores), columns=PRIMARY_LABELS)
score_df.insert(0, 'row_id', meta_test['row_id'].values)
score_df[PRIMARY_LABELS] = score_df[PRIMARY_LABELS].astype(np.float32)
score_df = score_df.drop_duplicates(subset='row_id', keep='last')

submission = sample_sub[['row_id']].merge(score_df, on='row_id', how='left')
assert submission.columns.tolist() == ['row_id'] + PRIMARY_LABELS

missing_mask = submission[PRIMARY_LABELS].isna().any(axis=1)
missing_count = int(missing_mask.sum())
if missing_count:
    print(f'Warning: {missing_count} row_ids are missing predictions after merge.')
    if used_test_fallback:
        print('This run used train fallback files, so missing row_ids are expected in editor mode.')
    submission[PRIMARY_LABELS] = submission[PRIMARY_LABELS].fillna(0.0)

submission[PRIMARY_LABELS] = submission[PRIMARY_LABELS].astype(np.float32)

submission.to_csv('/kaggle/working/submission.csv', index=False)
submission.to_csv('/kaggle/working/submission_exp_003_perch.csv', index=False)

print('Saved /kaggle/working/submission.csv')
print('Submission shape:', submission.shape)
print('Missing row_ids filled with zeros:', missing_count)
print(submission.iloc[:3, :8])


## What To Do On Kaggle

- Attach the four required inputs.
- If the notebook cannot find wheels, `perch_meta`, or the model automatically, fill the corresponding `*_DATASET_HINT` values in Cell 0.
- Keep internet access disabled.
- Run the notebook top to bottom and submit the generated `submission.csv`.
